# Analyse Big Data des accidents de la route avec PySpark

**Auteur :** Najib Mounsif

**Environnement :** Ubuntu 24.04 · Python 3.12 · OpenJDK 21 · PySpark 4

*Travail réalisé à partir du tutoriel de Rakesh Nain, « Big data Analysis of Road Crash Data using PySpark with PySpark Tutorial » (Medium / GitHub).*

## Problème
On dispose des données des accidents de la route en **Australie-Méridionale de 2012 à 2019** : environ 128 000 accidents et 273 000 véhicules impliqués, répartis dans 16 fichiers CSV. On veut :
1. charger et explorer ces données avec **PySpark** ;
2. répondre à des questions d'analyse (âge des conducteurs, accidents les plus graves, rôle de l'alcool et des drogues…) ;
3. comparer les **trois façons de travailler avec Spark** : les **RDD**, les **DataFrames** et **SparkSQL**, en vérifiant qu'elles donnent les mêmes résultats.

## Plan
1. Les données
2. Mise en place de Spark
3. Analyse avec les RDD
4. Analyse avec les DataFrames
5. Gravité des accidents : alcool et drogues
6. Une même requête avec RDD, DataFrame et SparkSQL
7. Conclusion

## 1. Les données

Les données sont publiées par le *Department of Planning, Transport and Infrastructure* (DPTI) d'Australie-Méridionale sur [data.sa.gov.au](https://data.sa.gov.au). On utilise deux tables, avec un fichier CSV par année :

| Table | Une ligne = | Colonnes utiles |
|---|---|---|
| **Crash** (`crash_data/`) | un accident | `REPORT_ID`, `Suburb`, `Total Cas` (blessés), `Total Fats` (tués), `Crash Type`, `CSEF Severity` (gravité), `DUI Involved` (alcool), `Drugs Involved` (drogues), `Year`, `Month`, `Day`, `Time` |
| **Units** (`units_data/`) | un véhicule (ou obstacle) impliqué | `REPORT_ID`, `Veh Reg State` (État d'immatriculation), `Unit Type`, `Veh Year`, `Sex`, `Age`, `Licence Type` |

Les deux tables sont reliées par la colonne **`REPORT_ID`** : un accident peut impliquer plusieurs véhicules.

La gravité `CSEF Severity` a quatre niveaux : `1: PDO` (dégâts matériels seulement), `2: MI` (blessé léger), `3: SI` (blessé grave), `4: Fatal` (mortel). Pour des raisons de confidentialité, le jour exact de l'accident n'est pas publié : seul le jour de la semaine est donné.

La cellule suivante récupère les données depuis le dépôt GitHub du tutoriel si elles ne sont pas déjà présentes.

In [ ]:
import os, subprocess, glob

DEPOT = 'https://github.com/RakeshNain/Big-data-Analysis-of-Road-Crash-Data-using-PySpark-with-PySpark-Tutorial.git'
DOSSIER = 'donnees_accidents'

if not os.path.isdir(DOSSIER):
    subprocess.run(['git', 'clone', '--depth', '1', DEPOT, DOSSIER], check=True)

CRASH = os.path.abspath(f'{DOSSIER}/crash_data')
UNITS = os.path.abspath(f'{DOSSIER}/units_data')
for nom, dossier in [('Crash', CRASH), ('Units', UNITS)]:
    fichiers = sorted(glob.glob(f'{dossier}/*.csv'))
    taille = sum(os.path.getsize(f) for f in fichiers) / 1e6
    print(f"{nom:<6}: {len(fichiers)} fichiers, {taille:.1f} Mo")

## 2. Mise en place de Spark

**PySpark** est l'interface Python d'Apache Spark, un moteur de calcul qui découpe les données en **partitions** et applique les traitements sur chaque partition en parallèle.

On travaille ici en **calcul parallèle** sur une seule machine (`local[*]` : un thread par cœur du processeur), et non en **calcul distribué** sur plusieurs machines. Le code est le même dans les deux cas : seule l'adresse du *master* change (par exemple `spark://serveur:7077` pour un cluster).

- `SparkConf` contient la configuration de l'application ;
- `SparkSession` est le point d'entrée pour les DataFrames et SparkSQL ;
- `SparkContext` (`spark.sparkContext`) est le point d'entrée pour les RDD.

In [ ]:
import csv, time
from pyspark import SparkConf
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
import pandas as pd
import matplotlib.pyplot as plt

conf = (SparkConf()
        .setMaster('local[*]')
        .setAppName('Analyse des accidents de la route')
        .set('spark.ui.showConsoleProgress', 'false'))

spark = SparkSession.builder.config(conf=conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel('ERROR')
print(f"Spark {spark.version} | master : {sc.master} | {sc.defaultParallelism} threads")

## 3. Analyse avec les RDD

Un **RDD** (*Resilient Distributed Dataset*) est la structure de base de Spark : une collection d'objets Python **immuable**, découpée en partitions traitées en parallèle. On l'utilise avec deux types d'opérations :
- les **transformations** (`map`, `filter`, `reduceByKey`…) créent un nouveau RDD. Elles sont **paresseuses** : rien n'est calculé tout de suite ;
- les **actions** (`count`, `collect`, `take`…) déclenchent le calcul et renvoient un résultat.

### 3.1 Créer un RDD
À partir d'une liste Python avec `parallelize`, en choisissant éventuellement le nombre de partitions :

In [ ]:
nombres = sc.parallelize(range(10))
print('Partitions par défaut :', nombres.getNumPartitions())

nombres = sc.parallelize(range(10), 5)
print('Partitions choisies   :', nombres.getNumPartitions())
print('Contenu de chaque partition :', nombres.glom().collect())
print('Somme des carrés :', nombres.map(lambda x: x * x).reduce(lambda a, b: a + b))

À partir de fichiers avec `textFile` : chaque **ligne** devient un élément (une chaîne de caractères) du RDD. Un motif `*.csv` permet de charger les 8 années d'un coup.

In [ ]:
crash_lignes = sc.textFile(f'{CRASH}/*.csv')
units_lignes = sc.textFile(f'{UNITS}/*.csv')
print(crash_lignes.take(2))

### 3.2 Nettoyage : en-têtes et découpage des lignes
Chaque fichier commence par une ligne d'en-tête : comme les 8 fichiers ont le même en-tête, on le récupère avec `first()` puis on le retire avec `filter`.

Il faut ensuite découper chaque ligne en colonnes. Le tutoriel original utilise `ligne.split(',')`, mais certaines valeurs entre guillemets contiennent elles-mêmes une virgule (par exemple le nom de commune `"CC OF NORWOOD,PAYNEHAM & ST PETERS"` dans la table Crash), ce qui décale toutes les colonnes. On utilise donc le module `csv` de Python, qui gère correctement les guillemets. On l'applique avec `mapPartitions`, partition par partition.

In [ ]:
def lire_csv(lignes):
    """Découpe correctement des lignes CSV (gère les virgules entre guillemets)."""
    return csv.reader(lignes)

crash_entete = next(csv.reader([crash_lignes.first()]))
units_entete = next(csv.reader([units_lignes.first()]))

crash_rdd = (crash_lignes.filter(lambda l: not l.startswith('"REPORT_ID"'))
                         .mapPartitions(lire_csv))
units_rdd = (units_lignes.filter(lambda l: not l.startswith('"REPORT_ID"'))
                         .mapPartitions(lire_csv))

# Positions des colonnes, pour ne pas manipuler des numéros en dur
C_ = {nom: i for i, nom in enumerate(crash_entete)}
U_ = {nom: i for i, nom in enumerate(units_entete)}

print('Accidents :', crash_rdd.count())
print('Véhicules :', units_rdd.count())
print('\nPremier véhicule :', dict(zip(units_entete, units_rdd.first())))

In [ ]:
# Vérification : toutes les lignes ont bien le bon nombre de colonnes
print('Lignes Crash mal découpées :', crash_rdd.filter(lambda r: len(r) != len(crash_entete)).count())
print('Lignes Units mal découpées :', units_rdd.filter(lambda r: len(r) != len(units_entete)).count())

# Avec split(',') comme dans le tutoriel original :
mal_coupees = crash_lignes.filter(lambda l: not l.startswith('"REPORT_ID"'))                           .filter(lambda l: len(l.split(',')) != len(crash_entete))
print("Lignes Crash mal découpées avec split(',') :", mal_coupees.count())
print('Exemple de valeur avec une virgule :', next(csv.reader([mal_coupees.first()]))[C_['LGA Name']])

### 3.3 Partitionnement
Par défaut, Spark découpe les fichiers en partitions de taille à peu près égale (ici, en gros une par fichier) :

In [ ]:
def afficher_partitions(rdd):
    tailles = rdd.glom().map(len).collect()   # glom : regroupe chaque partition dans une liste
    print(f"{len(tailles)} partitions :", tailles)

afficher_partitions(units_rdd)

On peut aussi choisir **comment** répartir les données. Par exemple, on sépare les véhicules immatriculés en **Australie-Méridionale (SA)** de tous les autres. Pour cela :
1. on transforme le RDD en paires **(clé, valeur)**, avec l'État d'immatriculation `Veh Reg State` comme clé ;
2. on appelle `partitionBy` avec une fonction qui renvoie le numéro de partition de chaque clé.

*(Le tutoriel original parle de la colonne `Lic State`, mais il utilise en réalité la 4ᵉ colonne, `Veh Reg State`.)*

In [ ]:
etat_rdd = units_rdd.map(lambda r: (r[U_['Veh Reg State']], r))

def partition_sa(etat):
    return 0 if etat == 'SA' else 1

etat_partitionne = etat_rdd.partitionBy(2, partition_sa)
afficher_partitions(etat_partitionne)

Sans surprise, la grande majorité des véhicules sont immatriculés en Australie-Méridionale, puisque ce sont des accidents survenus dans cet État. Ce type de partitionnement est utile quand on fait ensuite beaucoup de traitements par clé : toutes les lignes d'une même clé sont déjà au même endroit.

### 3.4 Requête : âge moyen des conducteurs par sexe
On garde les véhicules dont le sexe et l'âge sont connus, puis on calcule la moyenne par sexe.

Plutôt que `groupByKey` (qui envoie toutes les valeurs de chaque clé au même endroit), on utilise `reduceByKey` avec des paires **(somme, nombre)** : chaque partition calcule d'abord ses propres sommes, et seuls ces petits résultats sont échangés.

In [ ]:
def sexe_age(r):
    sexe, age = r[U_['Sex']], r[U_['Age']]
    if sexe in ('Male', 'Female') and age.isdigit():
        return [(sexe, (int(age), 1))]
    return []

age_moyen = (units_rdd.flatMap(sexe_age)
                      .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
                      .mapValues(lambda s: round(s[0] / s[1], 2)))
age_moyen.collect()

L'âge moyen des conducteurs impliqués dans un accident est presque le même pour les hommes et pour les femmes (environ 40 ans).

### 3.5 Requête : véhicules les plus anciens et les plus récents
On affiche l'année, l'État d'immatriculation et le type des 5 véhicules les plus anciens et des 5 plus récents. `takeOrdered` donne directement les *n* premiers éléments triés, sans trier ni rapatrier tout le RDD comme le ferait `sortByKey().collect()`.

In [ ]:
annees = (units_rdd.filter(lambda r: r[U_['Veh Year']].isdigit())
                  .map(lambda r: (int(r[U_['Veh Year']]), r[U_['Veh Reg State']], r[U_['Unit Type']])))

print('5 véhicules les plus anciens :')
for v in annees.takeOrdered(5, key=lambda v: v[0]):
    print('  ', v)
print('5 véhicules les plus récents :')
for v in annees.takeOrdered(5, key=lambda v: -v[0]):
    print('  ', v)

## 4. Analyse avec les DataFrames

Un **DataFrame** Spark est une table avec des **colonnes nommées et typées**, comme une table SQL ou un DataFrame pandas, mais découpée en partitions. Par rapport aux RDD :
- le code est plus court et plus lisible ;
- Spark connaît la structure des données, et son optimiseur (**Catalyst**) peut réorganiser les calculs pour les rendre plus rapides.

### 4.1 Créer un DataFrame
À partir d'une liste Python :

In [ ]:
exemple = spark.createDataFrame(
    [(1, 'Alice', 34), (2, 'Bilal', 27), (3, 'Chloé', 45)],
    ['id', 'prenom', 'age'])
exemple.show()
exemple.printSchema()

À partir des fichiers CSV. L'option `escape` indique comment sont échappés les guillemets à l'intérieur des valeurs. On convertit ensuite en nombres les colonnes dont on aura besoin pour les calculs avec `try_cast`, qui remplace les valeurs invalides (comme l'âge `'XXX'`, non communiqué) par `null` au lieu de provoquer une erreur. On retire aussi les espaces parasites (par exemple `'Provisional 1 '`).

In [ ]:
def lire(dossier):
    return (spark.read.option('header', True).option('escape', '"')
                 .csv(f'{dossier}/*.csv'))

crash_df = (lire(CRASH)
            .withColumn('Total Cas', F.expr('try_cast(`Total Cas` AS INT)'))
            .withColumn('Total Fats', F.expr('try_cast(`Total Fats` AS INT)')))

units_df = (lire(UNITS)
            .withColumn('Age', F.expr('try_cast(Age AS INT)'))      # 'XXX' devient null
            .withColumn('Licence Type', F.trim('Licence Type')))

print('Accidents :', crash_df.count(), '| Véhicules :', units_df.count())
crash_df.select('REPORT_ID', 'Suburb', 'Total Cas', 'Total Fats', 'Crash Type', 'CSEF Severity').show(3, truncate=False)

### 4.2 Accidents à Adelaide avec plus de 3 blessés

In [ ]:
adelaide = crash_df.filter((F.col('Suburb') == 'ADELAIDE') & (F.col('Total Cas') > 3))
adelaide.select('REPORT_ID', 'Year', 'Month', 'Day', 'Time', 'Total Cas', 'Crash Type').show(truncate=False)

### 4.3 Les 10 accidents avec le plus de blessés

In [ ]:
(crash_df.orderBy(F.desc('Total Cas'))
         .select('REPORT_ID', 'Suburb', 'Year', 'Total Cas', 'Total Fats', 'Crash Type')
         .show(10, truncate=False))

### 4.4 Nombre de tués par type d'accident

In [ ]:
tues_par_type = (crash_df.groupBy('Crash Type')
                         .agg(F.sum('Total Fats').alias('Tués'), F.count('*').alias('Accidents'))
                         .orderBy(F.desc('Tués')))
tues_par_type.show(truncate=False)

Les chocs contre un obstacle fixe (*Hit Fixed Object* : arbre, poteau…) font le plus de morts au total. Rapporté au nombre d'accidents, ce sont les collisions frontales (*Head On*) les plus meurtrières : 136 tués pour 2 086 accidents. À l'inverse, les collisions par l'arrière (*Rear End*) sont les plus nombreuses mais tuent très peu.

### 4.5 Blessés par quartier quand un conducteur sans permis est impliqué
Il faut **joindre** les deux tables : le type de permis est dans `Units`, le quartier et le nombre de blessés sont dans `Crash`.

Attention à un piège : un accident peut impliquer **plusieurs** véhicules sans permis. Avec une jointure classique (`inner`), comme dans le tutoriel original, ses blessés sont alors comptés plusieurs fois. Avec une **semi-jointure** (`left_semi`), on garde chaque accident une seule fois s'il implique au moins un conducteur sans permis.

In [ ]:
sans_permis = units_df.filter(F.col('Licence Type') == 'Unlicenced')

# Version du tutoriel : jointure classique (un accident peut être compté plusieurs fois)
par_quartier_inner = (crash_df.join(sans_permis, 'REPORT_ID', 'inner')
                              .groupBy('Suburb').agg(F.sum('Total Cas').alias('Blessés (inner)')))

# Version corrigée : chaque accident compté une seule fois
par_quartier = (crash_df.join(sans_permis, 'REPORT_ID', 'left_semi')
                        .groupBy('Suburb').agg(F.sum('Total Cas').alias('Blessés'),
                                               F.count('*').alias('Accidents')))

(par_quartier.join(par_quartier_inner, 'Suburb')
             .orderBy(F.desc('Blessés'))
             .show(10, truncate=False))

In [ ]:
doublons = crash_df.join(sans_permis, 'REPORT_ID').count() - crash_df.join(sans_permis, 'REPORT_ID', 'left_semi').count()
print(f"Lignes en double créées par la jointure classique : {doublons}")

## 5. Gravité des accidents : alcool et drogues

Les colonnes `DUI Involved` (alcool) et `Drugs Involved` (drogues) valent `Y` quand un conducteur a été contrôlé positif. On veut savoir si les accidents sont **plus graves** dans ces cas-là.

### 5.1 Nombre d'accidents par niveau de gravité

In [ ]:
crash_df.groupBy('CSEF Severity').count().orderBy('CSEF Severity').show()

### 5.2 Quatre scénarios
On compare la répartition des gravités dans quatre cas : drogues positif, alcool positif, les deux, aucun des deux.

Le tutoriel original écrit une fonction Python (UDF) différente pour chaque pourcentage. On fait plus simple et plus rapide avec une **fenêtre** (`Window`) : la somme sur la fenêtre donne le total de chaque scénario, et on divise chaque effectif par ce total, entièrement dans Spark.

In [ ]:
alcool = F.col('DUI Involved') == 'Y'
drogues = F.col('Drugs Involved') == 'Y'
scenario = (F.when(alcool & drogues, 'Alcool et drogues')
             .when(alcool, 'Alcool')
             .when(drogues, 'Drogues')
             .otherwise('Ni alcool ni drogues'))

# Un accident « alcool » ou « drogues » au sens du tutoriel inclut aussi les cas « les deux » :
# on construit donc les 4 scénarios du tutoriel par union.
cas = (crash_df.filter(drogues).withColumn('Scénario', F.lit('Drogues positif'))
       .unionByName(crash_df.filter(alcool).withColumn('Scénario', F.lit('Alcool positif')))
       .unionByName(crash_df.filter(alcool & drogues).withColumn('Scénario', F.lit('Les deux')))
       .unionByName(crash_df.filter(~F.coalesce(alcool, F.lit(False)) & ~F.coalesce(drogues, F.lit(False)))
                            .withColumn('Scénario', F.lit('Aucun des deux'))))

par_scenario = Window.partitionBy('Scénario')
gravite = (cas.groupBy('Scénario', 'CSEF Severity').count()
              .withColumn('%', F.round(100 * F.col('count') / F.sum('count').over(par_scenario), 2)))

gravite.orderBy('Scénario', 'CSEF Severity').show(20, truncate=False)

### 5.3 Tableau récapitulatif et graphiques
Pour tracer des graphiques, on convertit le résultat en DataFrame **pandas** avec `toPandas()`. C'est sans danger ici, car le résultat ne fait que 16 lignes. Il ne faut en revanche **jamais** faire `toPandas()` sur les données complètes : pandas charge tout dans la mémoire d'une seule machine, ce qui annule l'intérêt de Spark sur de gros volumes.

In [ ]:
ordre = ['Drogues positif', 'Alcool positif', 'Les deux', 'Aucun des deux']
g = gravite.toPandas()
effectifs = g.pivot(index='CSEF Severity', columns='Scénario', values='count')[ordre]
pourcents = g.pivot(index='CSEF Severity', columns='Scénario', values='%')[ordre]
display(effectifs)
display(pourcents)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
effectifs.plot(kind='bar', ax=ax1, rot=0)
ax1.set(title="Nombre d'accidents par gravité", xlabel='', ylabel="Nombre d'accidents", yscale='log')
pourcents.plot(kind='bar', ax=ax2, rot=0)
ax2.set(title='Répartition de la gravité dans chaque scénario', xlabel='', ylabel='% des accidents du scénario')
for ax in (ax1, ax2):
    ax.grid(axis='y', alpha=0.3); ax.legend(title='')
plt.tight_layout()
plt.savefig('gravite.png', dpi=120)
plt.show()

In [ ]:
graves = pourcents.loc[['3: SI', '4: Fatal']].sum()
print('Part des accidents graves ou mortels (SI + Fatal) :')
for s in ordre:
    print(f"  {s:<16}: {graves[s]:5.1f} %")
print(f"\nRisque relatif « les deux » / « aucun » : × {graves['Les deux'] / graves['Aucun des deux']:.1f}")

**Interprétation.** Sans alcool ni drogues, la grande majorité des accidents ne causent que des dégâts matériels (`1: PDO`), et seule une petite part est grave ou mortelle. Dès qu'un conducteur est positif à l'alcool ou aux drogues, la part des accidents graves (`3: SI`) et mortels (`4: Fatal`) augmente fortement, et elle est la plus élevée quand les deux sont combinés.

Attention : c'est une **corrélation**, pas forcément une causalité. Par exemple, les conducteurs sont plus souvent testés après un accident grave qu'après un simple accrochage, ce qui gonfle mécaniquement la part des accidents graves parmi les cas positifs.

## 6. Une même requête avec RDD, DataFrame et SparkSQL

**SparkSQL** permet d'interroger les DataFrames directement en SQL, après les avoir enregistrés comme **vues temporaires**. Les requêtes SQL et les DataFrames passent par le même optimiseur : ce sont deux syntaxes pour le même moteur.

On écrit deux requêtes des trois façons, puis on vérifie que les résultats sont identiques et on compare les temps.

In [ ]:
crash_df.createOrReplaceTempView('crash')
units_df.createOrReplaceTempView('units')

def chrono(fonction):
    t0 = time.perf_counter()
    resultat = fonction()
    return resultat, time.perf_counter() - t0

### 6.1 Requête 1 : détail des accidents à Adelaide
Pour chaque véhicule impliqué dans un accident à Adelaide : la date (année-mois-jour de la semaine), l'heure, le nombre de blessés de l'accident, le sexe, l'âge et le type de permis du conducteur. Les trois versions font une **jointure** entre les deux tables.

In [ ]:
def q1_rdd():
    accidents = (crash_rdd.filter(lambda r: r[C_['Suburb']] == 'ADELAIDE')
                          .map(lambda r: (r[C_['REPORT_ID']],
                                          (f"{r[C_['Year']]}-{r[C_['Month']]}-{r[C_['Day']]}",
                                           r[C_['Time']], int(r[C_['Total Cas']])))))
    vehicules = units_rdd.map(lambda r: (r[U_['REPORT_ID']],
                                         (r[U_['Sex']] or None,
                                          int(r[U_['Age']]) if r[U_['Age']].isdigit() else None,
                                          r[U_['Licence Type']].strip() or None)))
    return accidents.join(vehicules).map(lambda kv: kv[1][0] + kv[1][1]).collect()

def q1_df():
    return (crash_df.filter(F.col('Suburb') == 'ADELAIDE')
                    .join(units_df, 'REPORT_ID')
                    .select(F.concat_ws('-', 'Year', 'Month', 'Day').alias('Date'), 'Time',
                            'Total Cas', 'Sex', 'Age', 'Licence Type')
                    .collect())

def q1_sql():
    return spark.sql('''
        SELECT concat_ws('-', c.Year, c.Month, c.Day) AS Date, c.Time, c.`Total Cas`,
               u.Sex, u.Age, u.`Licence Type`
        FROM crash c JOIN units u ON c.REPORT_ID = u.REPORT_ID
        WHERE c.Suburb = 'ADELAIDE'
    ''').collect()

r_rdd, t_rdd = chrono(q1_rdd)
r_df,  t_df  = chrono(q1_df)
r_sql, t_sql = chrono(q1_sql)

def trier(lignes):
    # tri sur la version texte de chaque ligne (certaines valeurs sont None)
    return sorted((tuple(l) for l in lignes), key=lambda t: tuple(str(x) for x in t))
assert trier(r_rdd) == trier(r_df) == trier(r_sql)
print(f"{len(r_df)} lignes, identiques avec les 3 méthodes")
spark.createDataFrame(r_df[:5]).show(truncate=False)

### 6.2 Requête 2 : blessés par quartier avec un conducteur sans permis
La même requête qu'en 4.5 (version corrigée : chaque accident compté une seule fois), écrite des trois façons.

In [ ]:
def q2_rdd():
    accidents_sans_permis = (units_rdd.filter(lambda r: r[U_['Licence Type']].strip() == 'Unlicenced')
                                      .map(lambda r: (r[U_['REPORT_ID']], None))
                                      .distinct())
    return (crash_rdd.map(lambda r: (r[C_['REPORT_ID']], (r[C_['Suburb']], int(r[C_['Total Cas']]))))
                     .join(accidents_sans_permis)
                     .map(lambda kv: kv[1][0])
                     .reduceByKey(lambda a, b: a + b)
                     .collect())

def q2_df():
    return (crash_df.join(units_df.filter(F.col('Licence Type') == 'Unlicenced'), 'REPORT_ID', 'left_semi')
                    .groupBy('Suburb').agg(F.sum('Total Cas').alias('Blessés'))
                    .collect())

def q2_sql():
    return spark.sql('''
        SELECT Suburb, SUM(`Total Cas`) AS `Blessés`
        FROM crash
        WHERE REPORT_ID IN (SELECT REPORT_ID FROM units WHERE `Licence Type` = 'Unlicenced')
        GROUP BY Suburb
    ''').collect()

r2_rdd, t2_rdd = chrono(q2_rdd)
r2_df,  t2_df  = chrono(q2_df)
r2_sql, t2_sql = chrono(q2_sql)

assert trier(r2_rdd) == trier(r2_df) == trier(r2_sql)
print(f"{len(r2_df)} quartiers, identiques avec les 3 méthodes")
print('Top 5 :', sorted(r2_df, key=lambda l: -l[1])[:5])

### 6.3 Comparaison

In [ ]:
temps = pd.DataFrame({'RDD': [t_rdd, t2_rdd], 'DataFrame': [t_df, t2_df], 'SparkSQL': [t_sql, t2_sql]},
                     index=['Requête 1 (Adelaide)', 'Requête 2 (sans permis)']).round(2)
display(temps)

temps.T.plot(kind='bar', rot=0, figsize=(7, 3.5), title="Temps d'exécution (s)")
plt.ylabel('secondes'); plt.grid(axis='y', alpha=0.3); plt.tight_layout(); plt.show()

| | RDD | DataFrame | SparkSQL |
|---|---|---|---|
| Données | objets Python sans structure | colonnes nommées et typées | tables (vues) interrogées en SQL |
| Écriture | code Python ligne à ligne, positions de colonnes à gérer | méthodes chaînées (`filter`, `groupBy`, `join`…) | requête SQL classique |
| Optimisation | aucune : Spark exécute exactement ce qu'on écrit | optimiseur Catalyst | optimiseur Catalyst (même plan que le DataFrame) |
| Performances | les plus lentes : chaque ligne passe par Python | rapides : calcul dans la JVM | rapides, équivalentes aux DataFrames |
| Quand l'utiliser | données non structurées, contrôle fin des partitions | analyses de données structurées | équipes qui connaissent SQL, requêtes complexes |

Les DataFrames et SparkSQL sont en général nettement plus rapides que les RDD : avec un RDD, chaque ligne doit être envoyée à un processus Python pour exécuter nos `lambda`, alors que les DataFrames et SQL s'exécutent entièrement dans le moteur de Spark (JVM). Les écarts exacts dépendent de la machine.

## 7. Conclusion

**Sur les données :**
- la quasi-totalité des véhicules accidentés sont immatriculés en Australie-Méridionale ;
- l'âge moyen des conducteurs impliqués est presque identique pour les hommes et les femmes (environ 40 ans) ;
- les chocs contre un obstacle fixe font le plus de morts au total, et les collisions frontales sont les plus meurtrières par accident ;
- la part des accidents graves ou mortels est beaucoup plus élevée quand un conducteur est positif à l'alcool ou aux drogues, et maximale quand les deux sont combinés (corrélation, pas forcément causalité).

**Sur PySpark :**
- les **RDD** donnent un contrôle total (partitionnement personnalisé, traitements ligne par ligne), mais demandent plus de code et sont plus lents ;
- les **DataFrames** et **SparkSQL** sont plus concis, bénéficient de l'optimiseur Catalyst et donnent exactement les mêmes résultats ;
- les trois approches reposent sur le même principe : des données découpées en partitions, traitées en parallèle, avec des transformations paresseuses déclenchées par des actions.

**Corrections apportées par rapport au tutoriel original :**
- lecture CSV avec le module `csv` au lieu de `split(',')`, qui découpait mal plus de 21 000 lignes de la table Crash (toutes les colonnes après le nom de commune étaient décalées) ;
- colonne de partitionnement correctement nommée (`Veh Reg State`) ;
- `reduceByKey` et `takeOrdered` au lieu de `groupByKey` et `sortByKey().collect()`, plus efficaces ;
- semi-jointure pour ne pas compter plusieurs fois les blessés d'un même accident ;
- pourcentages calculés avec une fenêtre Spark au lieu de plusieurs UDF Python.

In [ ]:
spark.stop()